In [19]:
import threading
import time
from pathlib import Path

import torch
import numpy as np
import tqdm

from diff_mfld.geodesic.geodesic_funcs import ExpMethod, LogMethod
from diff_mfld.geometry.metric import MetricField
from diff_mfld.mfld import ComputeMfld
from optim.subsolver_methods import SubsolverMethod
from optim.subsolvers.rgd import RiemGradDescentCfg
from optim.subsolvers.rtr import RiemTrustRegionCfg
from problem import create_problem
from testing.testing_metrics import euclid_metric, scaled_metric, coupled_metric, asymmetric_metric

In [20]:
methods = [
    ((ExpMethod.IVP, LogMethod.BVP), "ivpbvp"),
    # ((ExpMethod.APPROX_O1, LogMethod.APPROX_O1), "o1"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O2), "o2"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O3), "o3"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O1), "o2_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O1), "o3_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O2), "o3_o2"),
]

# linear scaling of the domain
max_domain_scaling = [
    0.5, 1., 1.5, 2.
]

# parameters for random starting positions
p0_mean = np.array([2., -5., -8.])
p0_covar = 2. ** 2 * np.eye(3)
num_p0 = 20

# cost minimized at this point
cost_center = np.array([-5., -1., 3.])

# metrics to use in testing

metric_funcs = [
    (euclid_metric, "euclid"),
    (scaled_metric, "scaled"),
    (coupled_metric, "coupled"),
    (asymmetric_metric, "asymmetric"),
]


In [21]:
# generates the random points and scales them

# "To Everything? To the great Question of Life, the Universe and everything?"
r = np.random.default_rng(42)
p_rand_starting = r.multivariate_normal(p0_mean, p0_covar, num_p0)
max_starting_norm = np.max(np.linalg.vector_norm(p_rand_starting, axis=1))

# scales the optimization functions and starting points
p_rand_starting /= max_starting_norm
cost_center /= max_starting_norm

In [22]:
# configure the (unconstrained) configurations
optimizers = [
    ((SubsolverMethod.RIEM_GRAD_DESCENT, RiemGradDescentCfg()), "rgd"),
    ((SubsolverMethod.RIEM_TRUST_REGION, RiemTrustRegionCfg()), "rtr")
]

In [23]:
# root directory to store output files inside
base_data_dir = Path("../data/unconstrained")

In [ ]:
from diff_mfld.mfld import Mfld
import itertools

# run the optimization scheme

cfgs = itertools.product(
    metric_funcs, max_domain_scaling, optimizers, methods
)
cfg_pbar = tqdm.tqdm(cfgs, desc="Configurations")

for (
        (metric_fn, metric_fn_label),
        scaling,
        ((optimizer, optimizer_cfg), optimizer_label),
        ((exp_method, log_method), method_label)
) in cfg_pbar:

    # setup the manifold configuration
    metric = MetricField(lambda x1, x2, x3: metric_fn(x1, x2, x3, scaling=scaling))
    conn = metric.christoffels()  # Levi-Civita connection (metric-compatible and torsion-free)
    mfld = Mfld(metric, conn)

    compute_mfld = ComputeMfld(
        mfld=mfld,
        exp_method=exp_method,
        log_method=log_method,
        dist_method=log_method,  # which logarithm to use when computing distance
    )

    # scale the problem coordinates accordingly
    trials_p_starting = p_rand_starting * scaling
    trials_cost_center = cost_center * scaling

    # print(f"trials_cost_center: {trials_cost_center}")
    # assert False

    # create the problem
    cost, _ = create_problem(torch.tensor(trials_cost_center))

    trials_pbar = tqdm.tqdm(range(trials_p_starting.shape[0]), desc="Trials")
    for start_idx in trials_pbar:
        p0 = torch.tensor(trials_p_starting[start_idx])
        result = optimizer(cost, p0, compute_mfld, optimizer_cfg)
        print(result)

        # save the resulting dataclass
        filename = f"{optimizer_label}/metric_{metric_fn_label}__scaling_{scaling}__trial_{start_idx}__geod_method_{method_label}.pt"
        file_path = base_data_dir / filename
        print(f"\tSaving to {file_path}")

        torch.save(result, file_path)

Configurations: 0it [00:00, ?it/s]
Trials:   5%|▌         | 1/20 [00:01<00:33,  1.74s/it]

RiemGradDescentResult(success=True, p=tensor([-0.1521, -0.0522,  0.0771]), iters=7, history=RiemGradDescentHistory(p_hist=tensor([[ 0.0114, -0.1828, -0.1269],
        [-0.0442, -0.1384, -0.0575],
        [-0.0831, -0.1073, -0.0090],
        [-0.1103, -0.0855,  0.0250],
        [-0.1294, -0.0703,  0.0488],
        [-0.1427, -0.0596,  0.0655],
        [-0.1521, -0.0522,  0.0771]]), f_hist=tensor([0.0137, 0.0067, 0.0033, 0.0016, 0.0008, 0.0004, 0.0002])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_0.5__trial_0__geod_method_ivpbvp.pt



Trials:  10%|█         | 2/20 [00:03<00:33,  1.83s/it]

RiemGradDescentResult(success=True, p=tensor([-0.1561, -0.0506,  0.0770]), iters=8, history=RiemGradDescentHistory(p_hist=tensor([[ 0.0423, -0.2271, -0.2268],
        [-0.0225, -0.1694, -0.1275],
        [-0.0679, -0.1290, -0.0579],
        [-0.0997, -0.1007, -0.0093],
        [-0.1220, -0.0810,  0.0248],
        [-0.1375, -0.0671,  0.0487],
        [-0.1484, -0.0574,  0.0654],
        [-0.1561, -0.0506,  0.0770]]), f_hist=tensor([0.0242, 0.0118, 0.0058, 0.0028, 0.0014, 0.0007, 0.0003, 0.0002])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_0.5__trial_1__geod_method_ivpbvp.pt



Trials:  15%|█▌        | 3/20 [00:05<00:30,  1.82s/it]

...

RiemGradDescentResult(success=True, p=tensor([-0.6914, -0.1426,  0.4096]), iters=15, history=RiemGradDescentHistory(p_hist=tensor([[-0.1055, -0.6529, -0.7073],
        [-0.2825, -0.4988, -0.3699],
        [-0.4064, -0.3909, -0.1338],
        [-0.4931, -0.3153,  0.0315],
        [-0.5538, -0.2625,  0.1473],
        [-0.5963, -0.2255,  0.2283],
        [-0.6260, -0.1995,  0.2850],
        [-0.6469, -0.1814,  0.3247],
        [-0.6614, -0.1687,  0.3524],
        [-0.6716, -0.1598,  0.3719],
        [-0.6788, -0.1536,  0.3855],
        [-0.6838, -0.1493,  0.3950],
        [-0.6873, -0.1462,  0.4017],
        [-0.6897, -0.1441,  0.4064],
        [-0.6914, -0.1426,  0.4096]]), f_hist=tensor([3.7535e+00, 1.8392e+00, 9.0121e-01, 4.4159e-01, 2.1638e-01, 1.0603e-01,
        5.1953e-02, 2.5457e-02, 1.2474e-02, 6.1122e-03, 2.9950e-03, 1.4675e-03,
        7.1909e-04, 3.5236e-04, 1.7265e-04])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_2.0__trial_18__geod_method_ivpbvp.pt



Trials: 100%|██████████| 20/20 [01:21<00:00,  4.06s/it]
Configurations: 7it [05:56, 61.02s/it]

RiemGradDescentResult(success=True, p=tensor([-0.6889, -0.1429,  0.4113]), iters=15, history=RiemGradDescentHistory(p_hist=tensor([[ 0.2772, -0.6971, -0.4652],
        [-0.0146, -0.5297, -0.2004],
        [-0.2189, -0.4125, -0.0151],
        [-0.3618, -0.3305,  0.1146],
        [-0.4619, -0.2731,  0.2054],
        [-0.5320, -0.2329,  0.2690],
        [-0.5810, -0.2047,  0.3135],
        [-0.6153, -0.1850,  0.3446],
        [-0.6394, -0.1713,  0.3664],
        [-0.6562, -0.1616,  0.3817],
        [-0.6680, -0.1549,  0.3923],
        [-0.6762, -0.1501,  0.3998],
        [-0.6820, -0.1468,  0.4051],
        [-0.6860, -0.1445,  0.4087],
        [-0.6889, -0.1429,  0.4113]]), f_hist=tensor([4.0723e+00, 1.9954e+00, 9.7776e-01, 4.7910e-01, 2.3476e-01, 1.1503e-01,
        5.6366e-02, 2.7619e-02, 1.3533e-02, 6.6314e-03, 3.2494e-03, 1.5922e-03,
        7.8018e-04, 3.8229e-04, 1.8732e-04])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_2.0__trial_19__geod_method_ivpbvp.pt



Trials:   5%|▌         | 1/20 [00:04<01:22,  4.34s/it]

RiemTrustRegionResult(success=True, p=tensor([-0.6953, -0.1392,  0.4171]), iters=9, history=RiemTrustRegionHistory(p_hist=tensor([[ 0.2231, -0.8730, -0.7294],
        [ 0.0833, -0.7613, -0.5549],
        [-0.0565, -0.6496, -0.3803],
        [-0.1964, -0.5379, -0.2058],
        [-0.3362, -0.4261, -0.0312],
        [-0.4760, -0.3144,  0.1433],
        [-0.6158, -0.2027,  0.3179],
        [-0.6953, -0.1392,  0.4171],
        [-0.6953, -0.1392,  0.4171]]), f_hist=tensor([5.3946e+00, 3.8772e+00, 2.6099e+00, 1.5926e+00, 8.2521e-01, 3.0787e-01,
        4.0524e-02, 6.9126e-08, 6.9126e-08]), quality_hist=tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.0424]), radius_hist=tensor([0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.1250])))
	Saving to ../data/unconstrained/rtr/metric_euclid__scaling_2.0__trial_0__geod_method_ivpbvp.pt
